# Lab 4 — Stopword Removal
Solutions with code, outputs and written answers.

In [8]:
import re
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

stop_words = stopwords.words("english")
print(len(stop_words), "NLTK English stopwords")

198 NLTK English stopwords


## Exercise 1 — Identifying stopwords

In [9]:
text = "The student is working on a natural language processing project."
words = nltk.word_tokenize(text.lower())
print("Tokens:", words)
print("Stopwords found:", [word for word in words if word in stop_words])

Tokens: ['the', 'student', 'is', 'working', 'on', 'a', 'natural', 'language', 'processing', 'project', '.']
Stopwords found: ['the', 'is', 'on', 'a']


**Task 2 — the complement (non-stopwords)**

In [10]:
content_words = [w for w in words if w not in stop_words]
print(content_words)

['student', 'working', 'natural', 'language', 'processing', 'project', '.']


The list `['student', 'working', 'natural', 'language', 'processing', 'project', '.']` is enough for a human to guess the topic (a student working on a natural-language-processing project). The stopwords carry grammatical structure, not topic. The final `.` is not a stopword, because NLTK's list contains only words, so punctuation must be removed separately.

**Task 3 — are *natural*, *language*, *processing* stopwords?**

In [11]:
for w in ["natural", "language", "processing"]:
    print(f"{w!r:14} stopword? {w in stop_words}")

'natural'      stopword? False
'language'     stopword? False
'processing'   stopword? False


No, and that is expected. They are content words that define the topic. A generic list contains only function words that are frequent across all domains.

**Task 4 — first 15 words of NLTK's list, classified**

In [12]:
first15 = list(stopwords.words("english"))[:15]
print(f"{first15=}")

categories = {
    "Determiners / articles":   {"a", "an", "all", "any"},
    "Prepositions":             {"about", "above", "after", "against"},
    "Auxiliary / be-verbs":     {"am", "are", "aren", "aren't", "ain"},
    "Conjunctions":             {"and"},
    "Adverbs":                  {"again"},
}
categories = {k: [w for w in first15 if w in v] for k, v in categories.items()}
for k, v in categories.items():
    if v: print(f"{k:35} {v}")

first15=['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]
Determiners / articles              ['a', 'all', 'an', 'any']
Prepositions                        ['about', 'above', 'after', 'against']
Auxiliary / be-verbs                ['ain', 'am', 'are', 'aren', "aren't"]
Conjunctions                        ['and']
Adverbs                             ['again']


In this NLTK version the list is ordered alphabetically, so the first 15 entries are a mix of **determiners** (*a, an, all, any*), **prepositions** (*about, above, after, against*), **auxiliary/be-verbs** (*am, are, aren, aren't*, plus the clitic fragment *ain*), a **conjunction** (*and*) and an **adverb** (*again*). Elsewhere in the list you also find **pronouns** (*i, me, we, you, they*), further auxiliaries (*is, was, have, do*) and conjunctions (*but, or, if*). Note the fragments like *ain* and *aren*, which exist because of how contractions get split.

### Questions
1. **Stopwords in the example sentence:** *the, is, on, a*. (Stopwords found by the code above.)
2. **Are all frequent words useless?** No. *not* is extremely frequent but flips the meaning of a sentence. For sentiment analysis, "not good" and "good" are opposites. Likewise, in authorship attribution or stylometry, function words (*the, of, and*) are exactly the signal that distinguishes authors. Also, for phrase search, "to be or not to be" is made entirely of stopwords.
3. **Why does it depend on the application?** "Useless" means "carries no information for *this* task". A search engine indexing topics gains from dropping *the*, but a sentiment classifier, a question-answering system, a machine translator, or a language model needs function words, since they encode grammar and negation.

## Exercise 2 — Quantifying the effect of stopword removal

In [13]:
filtered = [word for word in words if word not in stop_words]
print("Original:", words)
print("Filtered:", filtered)

Original: ['the', 'student', 'is', 'working', 'on', 'a', 'natural', 'language', 'processing', 'project', '.']
Filtered: ['student', 'working', 'natural', 'language', 'processing', 'project', '.']


In [15]:
def stopword_stats(tokens, stop_words):
    original = len(tokens)
    remaining = len([t for t in tokens if t.lower() not in stop_words])
    removed = original - remaining
    pct = 100 * removed / original if original else 0.0
    return {"original": original, "remaining": remaining,
            "removed": removed, "percent_removed": round(pct, 1)}

In [16]:
samples = {
    "tweet-like":   "omg i just can't believe it is so good lol",
    "formal":       "The committee has decided that the proposal will be reviewed in the next meeting of the board.",
    "keyword list": "neural network optimization gradient descent backpropagation",
}
print(f"{'Style':14}{'orig':>6}{'remain':>8}{'removed':>9}{'% removed':>11}")
for style, s in samples.items():
    toks = [t for t in word_tokenize(s.lower()) if t.isalpha() or "'" in t]
    st = stopword_stats(toks, stop_words)
    print(f"{style:14}{st['original']:>6}{st['remaining']:>8}{st['removed']:>9}{st['percent_removed']:>11}")

Style           orig  remain  removed  % removed
tweet-like        11       6        5       45.5
formal            17       7       10       58.8
keyword list       6       6        0        0.0


**Comment.** The percentage varies strongly with style. Conversational and formal running text is dominated by function words, so a large share of tokens is removed (about 45% for the tweet-like sentence and about 59% for the formal one). A keyword list contains almost no function words, so almost nothing is removed (about 0%). The stopword ratio is a property of *register*: connected prose needs grammar words, while telegraphic or keyword text does not.

## Exercise 3 — The importance of context: negation

In [8]:
sentences = ["This movie is not good.",
             "This movie is really not that bad, actually."]

def remove_stopwords(sentence, sw=stop_words):
    toks = word_tokenize(sentence.lower())
    kept = [t for t in toks if t not in sw]
    dropped = [t for t in toks if t in sw]
    return toks, kept, dropped

for s in sentences:
    toks, kept, dropped = remove_stopwords(s)
    print(s)
    print("  after removal:", kept)
    print("  removed      :", dropped)
    print("  negation words removed:", [t for t in dropped if t in {"not","no","nor","never"} or t.endswith("n't")])

This movie is not good.
  after removal: ['movie', 'good', '.']
  removed      : ['this', 'is', 'not']
  negation words removed: ['not']
This movie is really not that bad, actually.
  after removal: ['movie', 'really', 'bad', ',', 'actually', '.']
  removed      : ['this', 'is', 'not', 'that']
  negation words removed: ['not']


In both sentences the negation **not** is removed (and with it, in sentence 2, the intensity-softening *that*). After removal we are left with `movie good` and `movie really bad actually`, so the first sentence now reads as plainly positive and the second as plainly negative, the *opposite* of what was meant.

In [9]:
POS = {"good", "great", "excellent"}
NEG = {"bad", "terrible", "awful"}

def naive_score(tokens):
    return sum(+1 for t in tokens if t in POS) + sum(-1 for t in tokens if t in NEG)

print(f"{'Sentence':48}{'before':>8}{'after':>8}")
for s in sentences:
    toks, kept, _ = remove_stopwords(s)
    print(f"{s:48}{naive_score(toks):>8}{naive_score(kept):>8}")

Sentence                                          before   after
This movie is not good.                                1       1
This movie is really not that bad, actually.          -1      -1


Four scores: sentence 1 gives **+1 before and +1 after**, and sentence 2 gives **-1 before and -1 after**. The naive scorer is blind to negation either way: stopword removal didn't change the score, because the scorer never looked at *not* in the first place. Both scores are wrong (the true sentiments are negative and mildly positive).

**Fix, step 1: keep the negations in the stopword list's complement.**

In [10]:
negations = {"not", "no", "nor", "never", "n't"}
custom_sw = stop_words - negations

for s in sentences:
    toks, kept, _ = remove_stopwords(s, custom_sw)
    print(s, "->", kept, "| naive score:", naive_score(kept))

This movie is not good. -> ['movie', 'not', 'good', '.'] | naive score: 1
This movie is really not that bad, actually. -> ['movie', 'really', 'not', 'bad', ',', 'actually', '.'] | naive score: -1


Keeping *not* alone **does not change the naive score**, since the scorer still only counts polarity words. The negation is now *available* but unused. The scorer must also use it. **Step 2:** flip the polarity of a sentiment word that follows a negation within a small window.

In [11]:
def negation_aware_score(tokens, window=3):
    score = 0
    for i, t in enumerate(tokens):
        pol = 1 if t in POS else -1 if t in NEG else 0
        if pol:
            if any(p in negations for p in tokens[max(0, i - window):i]):
                pol = -pol
            score += pol
    return score

for s in sentences:
    toks, kept_default, _ = remove_stopwords(s)             # default NLTK list
    _, kept_fixed, _ = remove_stopwords(s, custom_sw)       # negations preserved
    print(s)
    print("   default list      :", negation_aware_score(kept_default))
    print("   negations kept    :", negation_aware_score(kept_fixed))

This movie is not good.
   default list      : 1
   negations kept    : -1
This movie is really not that bad, actually.
   default list      : -1
   negations kept    : 1


With the default list, even a negation-aware scorer fails (the negation is gone): it gives +1 and -1. With *not* preserved, the scores become **-1** for "not good" (negative) and **+1** for "not that bad" (mildly positive), which is the sensible result. Both pieces are needed: **preserve the negation** and **use it**.

### Questions
1. **What disappears?** The polarity-flipping words (*not, no, never, n't*), hence the sentence's actual meaning; also hedges and intensifiers that are in the list (*very, too, that*).
2. **What happened to *not*?** Before the fix it was deleted by stopword removal; after the fix it stays in the token list, and, combined with a window-based rule, it flips the next sentiment word.
3. **Why problematic for sentiment?** Because negation is a high-frequency, high-impact function word. Standard lists treat it as noise, yet for sentiment analysis it can reverse the label. Blind removal systematically inverts the meaning of negated sentences.

## Exercise 4 — Custom, domain-specific stopword lists

In [12]:
custom_stopwords = {"the", "is", "a", "on"}

in_nltk_only = sorted(stop_words - custom_stopwords)
in_custom_only = sorted(custom_stopwords - stop_words)
print("Size NLTK:", len(stop_words), "| size custom:", len(custom_stopwords))
print("In custom but not NLTK:", in_custom_only)
print("In NLTK but not custom (first 20):", in_nltk_only[:20], "...")

Size NLTK: 198 | size custom: 4
In custom but not NLTK: []
In NLTK but not custom (first 20): ['about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before'] ...


The custom list is a strict subset of NLTK's list: nothing is in the custom list that is missing from NLTK, and NLTK contributes everything else (194 additional words).

In [13]:
corpus = [
    "Natural language processing helps computers understand human language.",
    "This course introduces NLP and text processing techniques.",
    "Tokenization is the first step in language processing pipelines.",
    "Students practise NLP exercises on stopword removal and normalization.",
    "Modern language models rely on large-scale text processing.",
]

all_tokens = [t for s in corpus for t in word_tokenize(s.lower()) if t.isalpha()]
freq = Counter(all_tokens)
print("Top 12 raw frequencies:")
for w, c in freq.most_common(12):
    print(f"  {w:14}{c}")

Top 12 raw frequencies:
  language      4
  processing    4
  nlp           2
  and           2
  text          2
  on            2
  natural       1
  helps         1
  computers     1
  understand    1
  human         1
  this          1


**Task 3 — extend the list with frequent domain words.** Among the most frequent words, the generic ones (*the, on, and, in...*) are already in NLTK. The top *non-stopword* items are domain words that appear everywhere and so cannot discriminate between documents.

In [14]:
top_non_stop = [(w, c) for w, c in freq.most_common() if w not in stop_words][:5]
print("Most frequent non-stopwords:", top_non_stop)

domain_words = {"language", "processing", "nlp"}   # appear in almost every sentence
extended_sw = stop_words | domain_words
print("Domain additions:", domain_words)

Most frequent non-stopwords: [('language', 4), ('processing', 4), ('nlp', 2), ('text', 2), ('natural', 1)]
Domain additions: {'processing', 'nlp', 'language'}


In [15]:
vocab_a = sorted({t for t in all_tokens if t not in stop_words})
vocab_b = sorted({t for t in all_tokens if t not in extended_sw})

print(f"(a) NLTK only            : {len(vocab_a)} words\n    {vocab_a}\n")
print(f"(b) NLTK + domain words  : {len(vocab_b)} words\n    {vocab_b}\n")
print("Removed by the domain additions:", sorted(set(vocab_a) - set(vocab_b)))

(a) NLTK only            : 25 words
    ['computers', 'course', 'exercises', 'first', 'helps', 'human', 'introduces', 'language', 'models', 'modern', 'natural', 'nlp', 'normalization', 'pipelines', 'practise', 'processing', 'rely', 'removal', 'step', 'stopword', 'students', 'techniques', 'text', 'tokenization', 'understand']

(b) NLTK + domain words  : 22 words
    ['computers', 'course', 'exercises', 'first', 'helps', 'human', 'introduces', 'models', 'modern', 'natural', 'normalization', 'pipelines', 'practise', 'rely', 'removal', 'step', 'stopword', 'students', 'techniques', 'text', 'tokenization', 'understand']

Removed by the domain additions: ['language', 'nlp', 'processing']


**Comparison.** List (a) is still dominated by *language*, *processing* and *nlp*, words that tell you nothing about which document you are in. List (b) removes them, leaving the words that actually distinguish documents (*tokenization*, *stopword*, *normalization*, *models*, ...).

### Question: why a custom list?
Frequency is relative to a corpus. A generic list is built from general English and knows nothing about your domain: in an NLP course collection, *processing* is as uninformative as *the* is in general text, while in a medical or legal corpus, entirely different words (*patient*, *court*) play that role. A domain-specific list can be used **in addition to** the generic one (as here) or **instead of** it when some generic stopwords are meaningful for the task (e.g. negations in sentiment analysis, or *the/who* in phrase search). A frequency-based approach (high term frequency across most documents, i.e. low IDF) is the standard way to discover them.

## Exercise 5 — Integrated preprocessing mini-pipeline

In [18]:
raw = ("Natural Language Processing (NLP) has grown VERY fast since 2018!! "
       "Companies such as Google and OpenAI now offer APIs at https://api.example.com, "
       "and models cost around $0.002 per 1,000 tokens. Isn't that amazing?")
print(raw)

Natural Language Processing (NLP) has grown VERY fast since 2018!! Companies such as Google and OpenAI now offer APIs at https://api.example.com, and models cost around $0.002 per 1,000 tokens. Isn't that amazing?


**Stage 1 — RegEx.** Mask the URL and the price with placeholders. The price pattern runs *before* any general number handling so that `$0.002` is masked as one unit. The placeholders use underscores so `word_tokenize` keeps each one as a single token.

In [20]:
URL_RE   = r"https?://[^\s,;]+"
PRICE_RE = r"\$\d+(?:,\d{3})*(?:\.\d+)?"

after_regex = re.sub(URL_RE, "__URL__", raw)
after_regex = re.sub(PRICE_RE, "__PRICE__", after_regex)
print(after_regex)

Natural Language Processing (NLP) has grown VERY fast since 2018!! Companies such as Google and OpenAI now offer APIs at __URL__, and models cost around __PRICE__ per 1,000 tokens. Isn't that amazing?


**Stage 2 — Tokenization** with `word_tokenize`.

In [25]:
tokens = word_tokenize(after_regex)
print(tokens)

['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'has', 'grown', 'VERY', 'fast', 'since', '2018', '!', '!', 'Companies', 'such', 'as', 'Google', 'and', 'OpenAI', 'now', 'offer', 'APIs', 'at', '__URL__', ',', 'and', 'models', 'cost', 'around', '__PRICE__', 'per', '1,000', 'tokens', '.', 'Is', "n't", 'that', 'amazing', '?']


Note how `Isn't` is split into `Is` + `n't`, and that placeholders and the numbers `2018` / `1,000` survive as single tokens.

**Stage 3 — Normalization:** lowercase, mask remaining numbers (`<num>`-style, here `__num__`), and expand the clitic `n't` to `not` so the negation is explicit.

In [19]:
def normalize(tok):
    t = tok.lower()
    if t == "n't":
        return "not"
    if re.fullmatch(r"\d+(?:,\d{3})*(?:\.\d+)?", t):
        return "__num__"
    return t

normalized = [normalize(t) for t in tokens]
print(normalized)

['natural', 'language', 'processing', '(', 'nlp', ')', 'has', 'grown', 'very', 'fast', 'since', '__num__', '!', '!', 'companies', 'such', 'as', 'google', 'and', 'openai', 'now', 'offer', 'apis', 'at', '__url__', ',', 'and', 'models', 'cost', 'around', '__price__', 'per', '__num__', 'tokens', '.', 'is', 'not', 'that', 'amazing', '?']


**Stage 4 — Stopword removal.** Decision on the negation in *Isn't*: **keep it.** Exercise 3 showed that deleting *not* can invert sentiment ("not good" becomes "good"). Here `is` is removed but `not` is kept (we take it out of the stopword list). Punctuation tokens are dropped in this stage too, since NLTK's list contains only words.

In [24]:
pipeline_sw = stop_words - {"not", "no", "nor", "never"}
punct = re.compile(r"^\W+$")

after_stop = [t for t in normalized if t not in pipeline_sw and not punct.match(t)]
print(after_stop)

['natural', 'language', 'processing', 'nlp', 'grown', 'fast', 'since', '__num__', 'companies', 'google', 'openai', 'offer', 'apis', '__url__', 'models', 'cost', 'around', '__price__', 'per', '__num__', 'tokens', 'not', 'amazing']


**Stage-by-stage view and summary table.**

In [23]:
stages = [
    ("raw",                    raw,           len(raw.split())),
    ("after RegEx",            after_regex,   len(after_regex.split())),
    ("after tokenization",     tokens,        len(tokens)),
    ("after normalization",    normalized,    len(normalized)),
    ("after stopword removal", after_stop,    len(after_stop)),
]
for name, rep, n in stages:
    print(f"--- {name} ({n}) ---")
    print(rep, "\n")

print(f"{'Stage':26}{'# tokens':>9}")
for name, _, n in stages:
    print(f"{name:26}{n:>9}")

--- raw (32) ---
Natural Language Processing (NLP) has grown VERY fast since 2018!! Companies such as Google and OpenAI now offer APIs at https://api.example.com, and models cost around $0.002 per 1,000 tokens. Isn't that amazing? 

--- after RegEx (32) ---
Natural Language Processing (NLP) has grown VERY fast since 2018!! Companies such as Google and OpenAI now offer APIs at __URL__, and models cost around __PRICE__ per 1,000 tokens. Isn't that amazing? 

--- after tokenization (40) ---
['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'has', 'grown', 'VERY', 'fast', 'since', '2018', '!', '!', 'Companies', 'such', 'as', 'Google', 'and', 'OpenAI', 'now', 'offer', 'APIs', 'at', '__URL__', ',', 'and', 'models', 'cost', 'around', '__PRICE__', 'per', '1,000', 'tokens', '.', 'Is', "n't", 'that', 'amazing', '?'] 

--- after normalization (40) ---
['natural', 'language', 'processing', '(', 'nlp', ')', 'has', 'grown', 'very', 'fast', 'since', '__num__', '!', '!', 'companies', 'such', 'as'

**Notes.** The raw/RegEx counts are whitespace-based (there are no tokens yet). The count rises at tokenization because punctuation and clitics become separate tokens, stays the same after normalization (it is a one-to-one mapping), and drops sharply after stopword removal (stopwords plus punctuation). The final representation, roughly `natural language processing nlp grown fast since __num__ companies google openai offer apis __url__ models cost around __price__ per __num__ tokens not amazing`, keeps the content words, the masked entities and the negation.